In [ ]:
import os

os.system("streamlit run app.py")

In [ ]:
!dir

In [1]:
!pip install streamlit

In [2]:
import streamlit
print(streamlit.__version__)

1.51.0


In [3]:
import sys
print(sys.executable)

C:\Users\saigi\anaconda3\python.exe


In [ ]:
import subprocess

subprocess.run(
    [sys.executable, "-m", "streamlit", "run", "app.py"]
)

In [ ]:
%%writefile app.py

import streamlit as st

st.title("Fraud Detection Dashboard")
st.success("Streamlit is working!")

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "streamlit", "run", "app.py"],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

print("\nRETURN CODE:")
print(result.returncode)

In [ ]:
import os
print(os.getcwd())

In [4]:
!pip install plotly

In [5]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
from PIL import Image

st.set_page_config(
    page_title="Credit Card Fraud Detection",
    layout="wide"
)
import joblib



df = pd.read_csv("creditcard.csv")


cart_model = joblib.load("cart_model.pkl")
rf_model = joblib.load("rf_model.pkl")
xgb_model = joblib.load("xgb_model.pkl")

scaler = joblib.load("scaler.pkl")
selected_features = joblib.load("selected_features.pkl")

x_test_scaled = joblib.load("x_test_scaled.pkl")
y_test = joblib.load("y_test.pkl")



st.sidebar.title("Navigation")

page = st.sidebar.radio(
    "Select Page",
    [
        "Overview",
        "Model Comparison",
        "Live Prediction",
        "Explainable AI",
        "Pipeline"
    ]
)



if page == "Overview":

    st.title("Credit Card Fraud Detection Dashboard")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Transactions",
        f"{len(df):,}"
    )

    col2.metric(
        "Fraud Cases",
        f"{df['Class'].sum():,}"
    )

    col3.metric(
        "Fraud Rate",
        f"{100*df['Class'].mean():.3f}%"
    )

    col4.metric(
        "Best ROC-AUC",
        "97.72%"
    )

    st.markdown("---")

    left, right = st.columns(2)

    with left:

        fig1 = px.pie(
            names=["Legitimate", "Fraud"],
            values=[
                (df["Class"] == 0).sum(),
                (df["Class"] == 1).sum()
            ],
            title="Fraud vs Legitimate Transactions"
        )

        st.plotly_chart(
            fig1,
            use_container_width=True,
            key="pie_overview"
        )

    with right:

        fig2 = px.histogram(
            df,
            x="Amount",
            nbins=18,
            title="Transaction Amount Distribution"
        )

        st.plotly_chart(
            fig2,
            use_container_width=True,
            key="amount_distribution"
        )

    st.subheader("Sample Transactions")

    st.dataframe(df.head(10))


elif page == "Model Comparison":
    st.title("Model Performance Comparison")
    
    metrics_df = pd.DataFrame({
        "Model": ["CART", "Random Forest", "XGBoost"],
        "Accuracy": [99.67, 99.93, 99.87],
        "Precision": [30.51, 79.79, 59.20],
        "Recall": [75.79, 78.95, 77.89],
        "F1 Score": [43.50, 79.37, 67.27],
        "ROC-AUC": [88.30, 97.72, 96.99]
    })
    
    color_map = {
        "CART": "#9b72cf",
        "Random Forest": "#00b4d8",
        "XGBoost": "#e0f2fe"
    }


    st.subheader("Overall Model Performance")
    
    metrics_long = pd.melt(
        metrics_df,
        id_vars="Model",
        value_vars=["Accuracy", "Precision", "Recall", "F1 Score"],
        var_name="Metric",
        value_name="Score"
    )
    
    fig_metrics = px.bar(
        metrics_long,
        x="Metric",
        y="Score",
        color="Model",
        barmode="group",
        color_discrete_map=color_map,
        text="Score",
        height=520,
        title="Accuracy, Precision, Recall and F1 Score Comparison"
    )
    
    fig_metrics.update_traces(
        texttemplate="%{text:.2f}",
        textposition="outside"
    )
    
    fig_metrics.update_layout(
        title_x=0.2,
        font_size=14,
        bargap=0.25,      
        bargroupgap=0.08  
    )
    
    st.plotly_chart(fig_metrics, use_container_width=True, key="combined_metrics")
    


    st.subheader("ROC-AUC Comparison")
    
    fig_auc = px.bar(
        metrics_df,
        x="Model",
        y="ROC-AUC",
        color="Model",
        color_discrete_map=color_map,
        text="ROC-AUC",
        height=500,
        title="ROC-AUC Score Comparison"
    )
    
    fig_auc.update_traces(
        texttemplate="%{text:.2f}",
        textposition="outside",
        width=0.35   
    )
    
    fig_auc.update_layout(
        title_x=0.3,
        font_size=14,
        bargap=0.4   
    )
    
    st.plotly_chart(fig_auc, use_container_width=True, key="roc_auc_chart")
    

    st.subheader("Confusion Matrices Comparison")
    
    confusion_data = {
        "CART":          {"TP": 164, "FP": 72,  "FN": 23,  "TN": 56487},
        "Random Forest": {"TP": 75,  "FP": 19,  "FN": 20,  "TN": 56632},
        "XGBoost":       {"TP": 74,  "FP": 51,  "FN": 21,  "TN": 56600},
    }


    st.subheader("Confusion Matrices Comparison")
    
    confusion_data = {
        "CART":          {"TP": 164, "FP": 72,  "FN": 23,  "TN": 56487},
        "Random Forest": {"TP": 75,  "FP": 19,  "FN": 20,  "TN": 56632},
        "XGBoost":       {"TP": 74,  "FP": 51,  "FN": 21,  "TN": 56600},
    }
    
    col1, col2, col3 = st.columns(3)
    
    for col, (model_name, data) in zip([col1, col2, col3], confusion_data.items()):
        with col:
            st.markdown(f"### {model_name}")
            st.markdown("---")
    
            # Row 1: TP and FP
            r1c1, r1c2 = st.columns(2)
            with r1c1:
                st.success(f"**TP**\n\n{data['TP']}")
            with r1c2:
                st.error(f"**FP**\n\n{data['FP']}")
    
            # Row 2: FN and TN
            r2c1, r2c2 = st.columns(2)
            with r2c1:
                st.error(f"**FN**\n\n{data['FN']}")
            with r2c2:
                st.success(f"**TN**\n\n{data['TN']}")


elif page == "Live Prediction":

    st.title("🚨 Live Fraud Detection")

    model_name = st.selectbox(
        "Select Model",
        ["CART", "Random Forest", "XGBoost"]
    )

    transaction_id = st.number_input(
        "Select Transaction Index",
        min_value=0,
        max_value=len(x_test_scaled)-1,
        value=0
    )

    if st.button("Predict Transaction"):

        sample = x_test_scaled[transaction_id].reshape(1, -1)


        if model_name == "CART":
            prediction = cart_model.predict(sample)[0]

        elif model_name == "Random Forest":
            prediction = rf_model.predict(sample)[0]

        else:
            prediction = xgb_model.predict(sample)[0]

        actual = y_test.iloc[transaction_id]

        st.subheader("Prediction Result")

        col1, col2 = st.columns(2)

        with col1:
            st.markdown("### Predicted")
            if prediction == 1:
                st.error("Fraud Transaction")
            else:
                st.success("Legitimate Transaction")

        with col2:
            st.markdown("### Actual")
            if actual == 1:
                st.error("⚠ Fraud Transaction")
            else:
                st.success("Legitimate Transaction")

        st.markdown("---")

        if prediction == actual:
            st.success("Correct Prediction")
        else:
            st.error("Incorrect Prediction")


elif page == "Explainable AI":

    st.title("Explainable AI")

    tab1, tab2, tab3 = st.tabs([
        "Random Forest",
        "XGBoost",
        "CART"
    ])
    
    with tab1:

        st.subheader("Random Forest Explainability")

        shap_img = Image.open("rf_shap.png")
        lime_img = Image.open("lime_rf.png")

        target_size = (800, 800)

        shap_img = shap_img.resize(target_size)
        lime_img = lime_img.resize(target_size)

    
        col1, col2 = st.columns(2)
    
        with col1:
            st.markdown("### SHAP Feature Importance")
            st.image(shap_img)
    
        with col2:
            st.markdown("### LIME Explanation")
            st.image(lime_img
            )

            
    with tab2:

        st.subheader("XGBoost Explainability")

        shap_img = Image.open("xgb_shap.png")
        lime_img = Image.open("lime_xgboost.png")

        target_size = (800, 800)

        shap_img1 = shap_img.resize(target_size)
        lime_img1 = lime_img.resize(target_size)

    
        col1, col2 = st.columns(2)
    
        with col1:
            st.markdown("### SHAP Feature Importance")
            st.image(shap_img1)
    
        with col2:
            st.markdown("### LIME Explanation")
            st.image(lime_img1
            )

    with tab3:

        st.subheader("CART Explainability")
    
        shap_img = Image.open("cart_shap.png")
        lime_img = Image.open("lime_cart.png")
    
        target_size = (800, 800)
    
        shap_img2 = shap_img.resize(target_size)
        lime_img2 = lime_img.resize(target_size)
    
        col1, col2 = st.columns(2)
    
        with col1:
            st.markdown("### SHAP Feature Importance")
            st.image(shap_img2)
    
        with col2:
            st.markdown("### LIME Explanation")
            st.image(lime_img2)
    
    
        



elif page == "Pipeline":

    st.title("Fraud Detection Pipeline")

    st.markdown("### End-to-End Machine Learning Workflow")

    st.markdown("---")
    st.subheader("1. Data Collection")

    st.write("""
    - Credit Card Transaction Dataset
    - Highly imbalanced real-world data
    - Target: Fraud vs Legitimate transactions
    """)

    st.markdown("---")
    st.subheader(" 2. Data Preprocessing")

    st.write("""
    - Handling missing values
    - Feature scaling (StandardScaler)
    - Removing irrelevant features
    """)

    st.markdown("---")
    st.subheader(" 3. Class Balancing")

    st.write("""
    - Applied SMOTE (Synthetic Minority Oversampling Technique)
    - Balanced fraud vs normal transactions
    - Improved model learning on minority class
    """)

    st.markdown("---")
    st.subheader(" 4. Feature Selection")

    st.write("""
    - Selected top 17 important features
    - Reduced noise and improved model performance
    """)

    st.markdown("---")
    st.subheader(" 5. Model Training")

    col1, col2, col3 = st.columns(3)

    with col1:
        st.success("CART")
        st.write("Simple, interpretable decision tree model")

    with col2:
        st.success("Random Forest")
        st.write("Ensemble model with high stability")

    with col3:
        st.success("XGBoost")
        st.write("Gradient boosting for high performance")

    st.markdown("---")

    st.subheader("6. Model Evaluation")

    st.write("""
    - Accuracy, Precision, Recall, F1-score
    - ROC-AUC comparison
    - Confusion Matrix analysis
    """)

    st.markdown("---")

    st.subheader("7. Explainable AI")

    st.write("""
    - SHAP: Feature importance explanation
    - LIME: Local prediction explanation
    - Model interpretability improved
    """)

    st.markdown("---")

 
    st.subheader("8. Deployment")

    st.write("""
    - Streamlit interactive dashboard
    - Real-time fraud prediction system
    - User-friendly ML interface
    """)

    st.markdown("---")

Overwriting app.py


In [ ]:
import joblib

x_test_scaled = joblib.load("x_test_scaled.pkl")
print(type(x_test_scaled))
print(x_test_scaled.shape)